# 📊 Credit Card Approval — Exploratory Data Analysis

This notebook performs a complete EDA on the UCI Credit Approval dataset.

**Dataset**: UCI Credit Screening (Australian credit applications)  
**Target**: ApprovalStatus — `+` (Approved) / `−` (Rejected)

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

matplotlib.rcParams['figure.figsize'] = (10, 6)
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False
sns.set_palette('Set2')

from preprocessing import load_raw_data

df = load_raw_data()
print(f'Shape: {df.shape}')
df.head()

## 1. Dataset Overview

In [ ]:
print('=== Column Types ===')
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Duplicates:', df.duplicated().sum())
print('\n=== Target Distribution ===')
print(df['ApprovalStatus'].value_counts())
df.describe()

## 2. Class Imbalance Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

counts = df['ApprovalStatus'].value_counts()
labels = ['Rejected (0)', 'Approved (1)']
colors = ['#ef4444', '#22c55e']

axes[0].bar(labels, counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold', pad=12)
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 3, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=labels, colors=colors, autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor':'white', 'linewidth':2})
axes[1].set_title('Approval Rate', fontsize=14, fontweight='bold', pad=12)

plt.suptitle('Target Variable — Credit Card Approval Status', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../static/images/plots/01_class_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

## 3. Numeric Feature Distributions (Histograms)

In [ ]:
numeric_cols = ['Age', 'Debt', 'YearsEmployed', 'CreditScore', 'Income']
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    ax = axes[i]
    approved = df[df['ApprovalStatus']==1][col].dropna()
    rejected = df[df['ApprovalStatus']==0][col].dropna()
    ax.hist(rejected, bins=25, alpha=0.65, color='#ef4444', label='Rejected', edgecolor='white')
    ax.hist(approved, bins=25, alpha=0.65, color='#22c55e', label='Approved', edgecolor='white')
    ax.set_title(f'{col} Distribution', fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    ax.legend()

axes[-1].axis('off')  # hide empty subplot
plt.suptitle('Numeric Feature Distributions by Approval Status', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../static/images/plots/02_numeric_distributions.png', bbox_inches='tight', dpi=150)
plt.show()

## 4. Boxplots — Outlier Detection

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 6))

for i, col in enumerate(numeric_cols):
    ax = axes[i]
    data = [df[df['ApprovalStatus']==0][col].dropna(),
            df[df['ApprovalStatus']==1][col].dropna()]
    bp = ax.boxplot(data, patch_artist=True, notch=False,
                    boxprops=dict(facecolor='#dbeafe'),
                    medianprops=dict(color='#1d4ed8', linewidth=2),
                    whiskerprops=dict(linewidth=1.5),
                    capprops=dict(linewidth=1.5))
    bp['boxes'][1].set_facecolor('#dcfce7')
    ax.set_xticklabels(['Rejected', 'Approved'], fontsize=9)
    ax.set_title(col, fontweight='bold')

plt.suptitle('Boxplots — Outlier Detection by Approval Status', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../static/images/plots/03_boxplots.png', bbox_inches='tight', dpi=150)
plt.show()

## 5. Correlation Heatmap

In [ ]:
numeric_df = df[numeric_cols + ['ApprovalStatus']].dropna()
corr = numeric_df.corr()

fig, ax = plt.subplots(figsize=(8, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix — Numeric Features', fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('../static/images/plots/04_correlation_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()

## 6. Categorical Feature Count Plots

In [ ]:
cat_cols = ['Gender', 'MaritalStatus', 'BankCustomer', 'Employed', 'PriorDefault', 'DriversLicense', 'Citizen']

fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    ax = axes[i]
    ct = pd.crosstab(df[col], df['ApprovalStatus'])
    ct.columns = ['Rejected', 'Approved']
    ct.plot(kind='bar', ax=ax, color=['#ef4444', '#22c55e'], edgecolor='white', linewidth=0.5)
    ax.set_title(f'{col}', fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=0)
    ax.legend(fontsize=8)

axes[-1].axis('off')
plt.suptitle('Categorical Features vs Approval Status', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../static/images/plots/05_categorical_features.png', bbox_inches='tight', dpi=150)
plt.show()

## 7. Income Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Income by approval status (log scale)
approved_income = df[df['ApprovalStatus']==1]['Income'].dropna()
rejected_income = df[df['ApprovalStatus']==0]['Income'].dropna()

axes[0].hist(np.log1p(rejected_income), bins=30, alpha=0.65, color='#ef4444', label='Rejected', edgecolor='white')
axes[0].hist(np.log1p(approved_income), bins=30, alpha=0.65, color='#22c55e', label='Approved', edgecolor='white')
axes[0].set_title('Income Distribution (log scale)', fontweight='bold')
axes[0].set_xlabel('log(Income + 1)')
axes[0].legend()

# Mean income by approval
mean_income = df.groupby('ApprovalStatus')['Income'].mean()
axes[1].bar(['Rejected', 'Approved'], mean_income.values, color=['#ef4444', '#22c55e'], edgecolor='white')
axes[1].set_title('Mean Income by Approval Status', fontweight='bold')
axes[1].set_ylabel('Mean Income')
for i, v in enumerate(mean_income.values):
    axes[1].text(i, v + 50, f'{v:.0f}', ha='center', fontweight='bold')

plt.suptitle('Income Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../static/images/plots/06_income_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

## 8. Credit History Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Prior Default vs Approval Rate
default_approval = df.groupby('PriorDefault')['ApprovalStatus'].mean() * 100
bars = axes[0].bar(default_approval.index, default_approval.values,
                   color=['#22c55e' if v > 50 else '#ef4444' for v in default_approval.values],
                   edgecolor='white')
axes[0].set_title('Approval Rate by Prior Default', fontweight='bold')
axes[0].set_xlabel('Prior Default (f=No, t=Yes)')
axes[0].set_ylabel('Approval Rate (%)')
axes[0].set_ylim(0, 105)
for bar, val in zip(bars, default_approval.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1f}%', ha='center', fontweight='bold')

# Credit Score distribution
axes[1].scatter(
    df[df['ApprovalStatus']==1]['CreditScore'], 
    df[df['ApprovalStatus']==1]['Income'],
    alpha=0.4, c='#22c55e', label='Approved', s=20
)
axes[1].scatter(
    df[df['ApprovalStatus']==0]['CreditScore'], 
    df[df['ApprovalStatus']==0]['Income'],
    alpha=0.4, c='#ef4444', label='Rejected', s=20
)
axes[1].set_title('Credit Score vs Income', fontweight='bold')
axes[1].set_xlabel('Credit Score')
axes[1].set_ylabel('Income')
axes[1].legend()

plt.suptitle('Credit History Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../static/images/plots/07_credit_history.png', bbox_inches='tight', dpi=150)
plt.show()

## 9. Employment Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

employed_rate = df.groupby('Employed')['ApprovalStatus'].mean() * 100
axes[0].bar(employed_rate.index, employed_rate.values,
            color=['#ef4444', '#22c55e'], edgecolor='white')
axes[0].set_title('Approval Rate by Employment Status', fontweight='bold')
axes[0].set_xlabel('Employed (f=No, t=Yes)')
axes[0].set_ylabel('Approval Rate (%)')
axes[0].set_ylim(0, 105)
for i, (idx, val) in enumerate(employed_rate.items()):
    axes[0].text(i, val + 1, f'{val:.1f}%', ha='center', fontweight='bold')

# Years Employed distribution
approved_ye = df[df['ApprovalStatus']==1]['YearsEmployed'].dropna()
rejected_ye = df[df['ApprovalStatus']==0]['YearsEmployed'].dropna()
axes[1].hist(rejected_ye, bins=20, alpha=0.65, color='#ef4444', label='Rejected', edgecolor='white')
axes[1].hist(approved_ye, bins=20, alpha=0.65, color='#22c55e', label='Approved', edgecolor='white')
axes[1].set_title('Years Employed Distribution', fontweight='bold')
axes[1].set_xlabel('Years Employed')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.suptitle('Employment Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../static/images/plots/08_employment_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

## 10. Missing Values Analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Count': missing, 'Percentage': missing_pct})
missing_df = missing_df[missing_df['Count'] > 0]

if len(missing_df) > 0:
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(missing_df.index, missing_df['Percentage'], color='#3b82f6', edgecolor='white')
    ax.set_title('Missing Values (%) by Feature', fontsize=13, fontweight='bold')
    ax.set_xlabel('Missing Percentage')
    for i, (idx, row) in enumerate(missing_df.iterrows()):
        ax.text(row['Percentage'] + 0.05, i, f"{row['Count']} ({row['Percentage']}%)", va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig('../static/images/plots/09_missing_values.png', bbox_inches='tight', dpi=150)
    plt.show()
else:
    print('No missing values found — all handled during load.')

## 11. Summary of Key Findings

| Finding | Insight |
|---|---|
| **Class Imbalance** | ~55.5% Rejected, ~44.5% Approved — mild imbalance handled with `class_weight='balanced'` |
| **Prior Default** | Strongest categorical predictor — applicants with prior defaults have drastically lower approval rates |
| **Income** | Higher income correlates with higher approval probability |
| **Credit Score** | Positive correlation with approval; outliers present (handled by StandardScaler) |
| **Employment** | Employed applicants have significantly higher approval rates |
| **Missing Values** | 37 rows with '?' in Gender, Age, MaritalStatus, etc. → imputed with median/mode |
| **Outliers** | Income and Debt have high-value outliers → StandardScaler is sufficient for tree models |